# **Policy learning: от оценки эффекта к правилу назначения**

Практика к модулю [«Гетерогенные эффекты»](https://ai-interpretability.school), урок «Как читать оценку и где ошибиться», шаг «От оценки к решению: policy learning».

Оценка $\hat\tau(x)$ говорит, на сколько воздействие меняет исход объекта. Решение отвечает на другой вопрос: кому назначать воздействие. В уроке его записывали правилом $\pi(x)\in\{0,1\}$ и выбирали по ценности $V(\pi)$ — среднему исходу, который получился бы при назначении по правилу; ценность оценивают дважды робастными оценками эффекта $\hat\Gamma_i$ ([Athey и Wager, 2021](https://arxiv.org/abs/1702.02896)). Здесь мы проделаем это на рандомизированном эксперименте с рассылкой писем — синтетическом, по образцу набора Hillstrom, чтобы истинный эффект был известен:

1. данные: кому отправили письмо и кто зашел на сайт;
2. дважды робастные оценки эффекта $\hat\Gamma_i$ с cross-fitting;
3. ценность простых правил и цена письма;
4. дерево правил `econml.policy.DRPolicyTree`: глубина, оценка ценности на обучающих и на отложенных данных;
5. сравнение с правилом «назначать, где оценка CATE выше цены» и правило с бюджетом;
6. где ломается: выбор исхода, по которому считают ценность, и неустойчивость дерева на малых выборках.

Тетрадь идет на процессоре; весь прогон занимает две-три минуты.

In [ ]:
!pip install -q econml==0.17.0

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, train_test_split
from econml.policy import DRPolicyTree
from econml.dml import CausalForestDML

warnings.filterwarnings('ignore', message='.*encountered in matmul')   # ложные предупреждения numpy на некоторых процессорах
SEED = 0
pd.set_option('display.width', 140)

## 1. Данные: синтетическая рассылка

Сюжет и признаки взяты из [MineThatData E-Mail Analytics Challenge](https://blog.minethatdata.com/2008/03/minethatdata-e-mail-analytics-and-data.html) (Kevin Hillstrom, 2008): клиентки и клиенты интернет-магазина, которые покупали в последние 12 месяцев, случайно разделены на группы с письмом и без письма, а через две недели записано, зашел ли человек на сайт (`visit`), купил ли (`conversion`) и сколько потратил (`spend`, доллары). Сам набор MineThatData опубликован без лицензии, поэтому тетрадь генерирует его аналог функцией `make_email_rct`: 42 000 строк, письмо с женской коллекцией ($T=1$) или без письма ($T=0$) с вероятностью 1/2. Доли групп, посещений и покупок близки к исходному набору, а истинный эффект письма известен: в синтетике его задает формула, и по ней можно проверить, нашло ли правило тех, кому письмо помогает.

Признаки:

- `recency` — сколько месяцев прошло с последней покупки;
- `history` — сколько потрачено за год, доллары;
- `mens`, `womens` — покупал ли товары мужской и женской коллекции;
- `newbie` — новый клиент за последние 12 месяцев;
- `zip_code` — тип района (городской, пригородный, сельский), `channel` — через что покупал (телефон, сайт, оба).

Исход для правила — посещение сайта `visit`. Половина данных идет на обучение правил, половина — на отложенную оценку их ценности.

In [ ]:
def make_email_rct(n=42000, seed=SEED):
    rng = np.random.default_rng(seed)
    recency = rng.integers(1, 13, n)
    history = np.round(np.exp(rng.normal(5.0, 0.9, n)), 2)
    mens, womens = rng.binomial(1, 0.55, n), rng.binomial(1, 0.55, n)
    none, pick = (mens + womens) == 0, rng.binomial(1, 0.5, n)
    mens = np.where(none & (pick == 1), 1, mens)
    womens = np.where(none & (pick == 0), 1, womens)
    newbie = rng.binomial(1, 0.5, n)
    zip_code = rng.choice(['Urban', 'Suburban', 'Rural'], n, p=[0.4, 0.45, 0.15])
    channel = rng.choice(['Web', 'Phone', 'Multichannel'], n, p=[0.44, 0.44, 0.12])
    T = rng.binomial(1, 0.5, n)
    p0 = 1 / (1 + np.exp(-(-2.4 + 0.25 * womens + 0.15 * mens - 0.06 * (recency - 6)
                           + 0.25 * np.log(history / 150) + 0.3 * (channel == 'Web') - 0.2 * newbie)))
    tau = 0.072 * womens + 0.012 * (1 - womens) - 0.002 * (recency - 6)   # истинный эффект письма на посещение
    visit = rng.binomial(1, np.clip(p0 + T * tau, 0, 1))
    conversion = visit * rng.binomial(1, 0.07, n)
    spend = np.round(conversion * np.exp(rng.normal(4.4, 0.6, n)), 2)
    df = pd.DataFrame(dict(recency=recency, history=history, mens=mens, womens=womens, zip_code=zip_code,
                           newbie=newbie, channel=channel, T=T, visit=visit, conversion=conversion, spend=spend))
    return df, tau


df, tau_true = make_email_rct()
T = df['T'].values
Xdf = pd.get_dummies(df[['recency', 'history', 'mens', 'womens', 'newbie', 'zip_code', 'channel']],
                     columns=['zip_code', 'channel']).astype(float)
X, names = Xdf.values, list(Xdf.columns)
Y = df.visit.values

tr, te = train_test_split(np.arange(len(df)), test_size=0.5, random_state=SEED, stratify=T)
print('клиентов в двух группах:', len(df), '| доля получивших письмо:', round(T.mean(), 3))
print('обучение:', len(tr), '| отложенная оценка:', len(te))
print('доля посетивших сайт: с письмом', round(Y[T == 1].mean(), 4), '| без письма', round(Y[T == 0].mean(), 4))
print('истинный эффект: средний', round(tau_true.mean(), 4), '| у покупавших женскую коллекцию', round(tau_true[df.womens == 1].mean(), 4), '| у остальных', round(tau_true[df.womens == 0].mean(), 4))


## 2. Дважды робастные оценки эффекта

Для каждого объекта строим оценку его эффекта

$$\hat\Gamma_i=\hat\mu_1(X_i)-\hat\mu_0(X_i)+\frac{T_i\,(Y_i-\hat\mu_1(X_i))}{\hat e(X_i)}-\frac{(1-T_i)\,(Y_i-\hat\mu_0(X_i))}{1-\hat e(X_i)},$$

где $\hat\mu_1$, $\hat\mu_0$ — модели исхода в двух группах, $\hat e$ — модель склонности. Это тот же псевдоисход, что у DR-learner. Каждая $\hat\Gamma_i$ по отдельности очень шумная, но ее среднее по любой группе объектов — несмещенная оценка среднего эффекта в этой группе, если верна хотя бы одна из двух моделей. Вспомогательные модели обучаем на других фолдах (cross-fitting), а оценки для обучающей и для отложенной половины считаем по отдельности: отложенные оценки не должны зависеть от моделей, видевших обучающие данные.

**Задание 1. Напишите функцию `dr_scores(Xs, Ts, Ys, seed)`, которая на пяти фолдах `KFold(5, shuffle=True, random_state=seed)` обучает на остальных фолдах `GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=SEED)` отдельно по группам $T=1$ и $T=0$ и `LogisticRegression(max_iter=1000)` для склонности и возвращает $\hat\Gamma_i$ объектов текущего фолда. Посчитайте `G_tr = dr_scores(X[tr], T[tr], Y[tr], 0)` и `G_te = dr_scores(X[te], T[te], Y[te], 1)` и сравните среднее `G_te` с разностью средних исходов на отложенной половине.**

In [ ]:
# Ваш код здесь

## 3. Ценность правила и цена письма

Если письмо ничего не стоит, а эффект почти у всех неотрицательный, лучшее правило — писать всем, и выбирать нечего. Сделаем задачу честной по отношению к бизнесу: пусть письмо стоит $c$ посещений — это может быть стоимость отправки, риск отписки или усталость от рассылок, выраженная в тех же единицах, что исход. Тогда выигрыш правила относительно «не писать никому» равен

$$\hat V_c(\pi)=\frac1n\sum_i\pi(X_i)\,(\hat\Gamma_i-c).$$

Возьмем $c=0.045$ — примерно средний эффект письма на посещение: при такой цене «писать всем» почти ничего не дает, и выигрыш появляется только у правила, которое отличает тех, кому письмо помогает.

**Задание 2. Напишите функцию `policy_value(pi, G, c=COST)` и посчитайте на отложенной половине выигрыш трех правил: «писать всем», «писать тем, кто покупал женскую коллекцию» (`womens == 1`) и «писать тем, кто не покупал мужскую» (`mens == 0`). Для каждого выведите долю получивших письмо и выигрыш.**

In [ ]:
COST = 0.045
col = lambda name, rows: X[rows][:, names.index(name)]

In [ ]:
# Ваш код здесь

## 4. Дерево правил

`econml.policy.DRPolicyTree` сам строит $\hat\Gamma_i$ по обучающим данным и ищет дерево заданной глубины, максимизирующее $\frac1n\sum_i\pi(X_i)\hat\Gamma_i$. Цену письма передаем через исход: если вычесть $c$ из исхода всех получивших письмо, эффект каждого уменьшится ровно на $c$.

Учим деревья глубины 1, 2, 3 и 4 (`min_samples_leaf=500`) и для каждого сравниваем выигрыш, посчитанный по обучающим оценкам `G_tr`, с выигрышем по отложенным `G_te`. Дерево ищет правило по своим внутренним оценкам $\hat\Gamma_i$, а `G_tr` посчитаны нашими моделями из задания 1, поэтому выигрыш по `G_tr` — тоже проверка, хотя и на тех же объектах.

In [ ]:
def policy_tree(depth, rows=tr, outcome=Y, c=COST, leaf=500):
    return DRPolicyTree(model_regression=GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=SEED),
                        model_propensity=LogisticRegression(max_iter=1000), max_depth=depth, min_samples_leaf=leaf,
                        cv=5, random_state=SEED).fit(outcome[rows] - c * T[rows], T[rows], X=X[rows])

trees = {d: policy_tree(d) for d in (1, 2, 3, 4)}
depth_tab = pd.DataFrame({d: {'доля получивших письмо': t.predict(X[te]).mean(),
                              'выигрыш по обучающим G_tr': policy_value(t.predict(X[tr]), G_tr),
                              'выигрыш по отложенным G_te': policy_value(t.predict(X[te]), G_te)}
                          for d, t in trees.items()}).T
depth_tab.index.name = 'глубина'
print(depth_tab.round(5))

Глубины 1, 2 и 3 дают одно и то же правило: писать тем, кто покупал женскую коллекцию. В синтетике видно, что это и есть лучшее правило: истинный эффект письма у покупавших женскую коллекцию 0.071, у остальных 0.011 (вывод раздела 1), а цена письма 0.045. Глубина 4 добавляет разбиения, и выигрыш на отложенной половине уменьшается: глубже не значит лучше, лишние разбиения ловят шум оценок $\hat\Gamma_i$.

Дерево глубины 2 целиком: в каждом узле — признак и порог, в листе — рекомендованное действие (0 — не писать, 1 — писать).

In [ ]:
plt.figure(figsize=(12, 5))
trees[2].plot(feature_names=names, treatment_names=['no e-mail', 'e-mail'], fontsize=9)
plt.show()

## 5. Правило по оценке CATE и правило с бюджетом

Другой путь к правилу — сначала оценить CATE, потом назначать там, где $\hat\tau(x)>c$. Оценим CATE причинным лесом `CausalForestDML` на обучающей половине и сравним выигрыш этого правила на отложенной половине с деревом глубины 2. Дерево читается целиком; правило по оценке CATE — нет: чтобы объяснить, почему письмо получил конкретный человек, придется объяснять сам лес.

Бюджет. Урок говорит, что ограничения входят в задачу. Пусть писем хватает только на 20% клиентов. Правило с бюджетом по оценке CATE — отправить письмо 20% клиентов с наибольшей $\hat\tau(x)$.

In [ ]:
cfp = CausalForestDML(model_y=GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=SEED),
                      model_t=GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=SEED),
                      discrete_treatment=True, n_estimators=500, min_samples_leaf=50, cv=5, random_state=SEED)
cfp.fit(Y[tr], T[tr], X=X[tr])
tau_te = cfp.effect(X[te])
pi_cate = (tau_te > COST).astype(float)
pi_tree = trees[2].predict(X[te]).astype(float)
print(f'правило «оценка CATE > c»: доля {pi_cate.mean():.3f}, выигрыш {policy_value(pi_cate, G_te):+.5f}')
print(f'дерево глубины 2:          доля {pi_tree.mean():.3f}, выигрыш {policy_value(pi_tree, G_te):+.5f}')
print(f'доля клиентов, по которым два правила согласны: {(pi_cate == pi_tree).mean():.3f}')

**Задание 3. Правило с бюджетом 20%: отправьте письмо 20% клиентов отложенной половины с наибольшей оценкой CATE `tau_te` и посчитайте прирост доли посетивших сайт на одного клиента всей отложенной половины, $\frac1n\sum_i\pi(X_i)\hat\Gamma_i$ (без цены письма: бюджет уже ограничивает число писем). Сравните со случайными 20% (среднее по 200 случайным выборкам с генератором `np.random.default_rng(SEED)`) и нарисуйте кривую: доля получивших письмо от 0 до 1 по оси X, прирост по оси Y, для порядка по `tau_te` и для случайного порядка.**

In [ ]:
# Ваш код здесь

## 6. Где ломается

### 6.1. Ценность считается по выбранному исходу

Урок называет это четвертой ловушкой: правило лучшее только для того исхода, по которому считали ценность. Обучим дерево глубины 2 на другом исходе — сумме покупок `spend` в долларах, с ценой письма, равной среднему эффекту письма на `spend` по обучающей половине, — и сравним, кому два правила отправляют письмо и какой выигрыш каждое дает по обоим исходам.

In [ ]:
S = df.spend.values
cost_spend = S[tr][T[tr] == 1].mean() - S[tr][T[tr] == 0].mean()
tree_spend = policy_tree(2, outcome=S, c=cost_spend)
pi_spend = tree_spend.predict(X[te]).astype(float)
G_te_spend = dr_scores(X[te], T[te], S[te], 1)
print(f'цена письма для spend: {cost_spend:.3f} доллара')
print(f'дерево по visit: доля {pi_tree.mean():.3f} | дерево по spend: доля {pi_spend.mean():.3f}')
print(f'доля клиентов, по которым два дерева согласны: {(pi_tree == pi_spend).mean():.3f}')
table = pd.DataFrame({'выигрыш по visit': [policy_value(p, G_te) for p in (pi_tree, pi_spend)],
                      'выигрыш по spend, доллары': [policy_value(p, G_te_spend, cost_spend) for p in (pi_tree, pi_spend)]},
                     index=['дерево по visit', 'дерево по spend'])
print(table.round(5))
plt.figure(figsize=(12, 5))
tree_spend.plot(feature_names=names, treatment_names=['no e-mail', 'e-mail'], fontsize=9)
plt.show()

Два дерева согласны только на 29% клиентов: дерево по `spend` пишет 5.7% клиентов, дерево по `visit` — 65.3%. Каждое дерево лучше по своему исходу: по `spend` дерево по `spend` дает 0.01537 доллара на клиента против 0.00049 у дерева по `visit`, а по посещениям уходит в минус, −0.00383. Покупку совершает около 1% клиентов, оценки $\hat\Gamma_i$ по `spend` намного шумнее, и правило по ним держится на малой группе. Выбор исхода — это и выбор цели, и выбор того, насколько шумную ценность придется оценивать.

### 6.2. Неустойчивость дерева на малой выборке

Гарантия Athey и Wager — о сожалении (regret): ценность найденного правила отстает от лучшего правила класса на величину порядка $1/\sqrt n$. На малой выборке это отставание велико, и дерево меняется от подвыборки к подвыборке.

**Задание 4. Для размеров подвыборки 500, 2000 и 8000 объектов обучающей половины и сидов 0–9 обучите дерево глубины 2 (`policy_tree(2, rows=...)`, `leaf=50`; подвыборку берите `np.random.default_rng(seed).choice(tr, size, replace=False)`). Для каждого размера выведите среднее и стандартное отклонение выигрыша на отложенной половине и среднюю долю клиентов отложенной половины, по которым дерево согласно с деревом `trees[2]`, обученным на всей обучающей половине.**

In [ ]:
# Ваш код здесь

## Реализации

- `econml.policy.DRPolicyTree` и `econml.policy.DRPolicyForest` — дерево и лес правил на дважды робастных оценках; цену и бюджет напрямую не принимают, цену передают через исход, как здесь.
- R-пакет [policytree](https://grf-labs.github.io/policytree/): `double_robust_scores()` строит $\hat\Gamma_i$ по причинному лесу из grf, `policy_tree()` ищет точно оптимальное дерево заданной глубины полным перебором.

## Что унести из тетради

- **Правило выбирают по ценности, а не по точности оценки эффекта.** Дважды робастные оценки $\hat\Gamma_i$ по отдельности шумные, но их среднее по группе — оценка эффекта в группе, и этого достаточно, чтобы сравнивать правила.
- **Без цены воздействия задачи нет.** При бесплатном письме и неотрицательном эффекте лучшее правило — писать всем.
- **Ценность правила оценивают на отложенных данных, и глубже не значит лучше.** Здесь все решает одно разбиение по `womens`; дерево глубины 4 на отложенной половине дает меньший выигрыш.
- **Правило зависит от исхода.** Деревья по посещениям и по покупкам согласны меньше чем на трети клиентов, и каждое лучше по своему исходу и хуже по чужому. Цель и цену нужно записать до поиска правила, а правило по шумному исходу (редкие покупки) держится на малой группе.
- **Малая выборка — неустойчивое правило.** Сожаление убывает как $1/\sqrt n$, и при сотнях объектов дерево меняется от подвыборки к подвыборке.

Источники:
1. [Athey & Wager, 2021. Policy learning with observational data](https://arxiv.org/abs/1702.02896)
2. [Hillstrom, 2008. The MineThatData E-Mail Analytics and Data Mining Challenge](https://blog.minethatdata.com/2008/03/minethatdata-e-mail-analytics-and-data.html) — образец сюжета и признаков синтетических данных
3. [EconML](https://github.com/py-why/EconML), [policytree](https://grf-labs.github.io/policytree/)